In [1]:
from datasets import load_dataset, load_from_disk
import json
import gzip
import os
from collections import Counter

In [2]:
# Load your local dataset
language = 'Java'

ds = load_dataset(
    "AISE-TUDelft/the-heap",
    f"{language}",
    split="train",
    num_proc=12
)

print(len(ds))

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/67 [00:00<?, ?it/s]

5168193


In [3]:
# Load the annotation file, optionally gzipped
annotations = []
with gzip.open("../../data/Java/satd_count_spans_annotations.jsonl.gz", "r") as f:
    for line in f:
        annotations.append(json.loads(line))

# Check length matches
assert len(ds) == len(annotations), "Length mismatch!"

def add_annotations(batch, indices):
    return {
        "satd_spans": [annotations[i]["satd_spans"] for i in indices],
        "satd_count": [annotations[i]["satd_count"] for i in indices],
    }

ds_all = ds.map(add_annotations, with_indices=True, batched=True, desc="Adding SATD spans and counts")

# # Extract columns
# satd_span = [a["satd_span"] for a in annotations]
# satd_count = [a["satd_count"] for a in annotations]

# # Add columns to the local dataset
# local_ds = ds.add_column("satd_span", satd_span)
# local_ds = ds.add_column("satd_count", satd_count)

In [ ]:
ds_sorted = ds_all.sort("satd_count", reverse=True)
amount = 1000
top100 = ds_sorted.select(range(amount))

# Create output directories
summary_path = f"./investigation/top{amount}_satd_summary.jsonl"
code_output_dir = f"./investigation/top{amount}_satd_files"
os.makedirs(code_output_dir, exist_ok=True)

with open(summary_path, "w", encoding="utf-8") as summary_file:
    for entry in top100:
        filename = entry["file_name"]
        text = entry["content"]
        count = entry["satd_count"]
        spans = entry["satd_spans"]

        # Get SATD comments from spans
        satd_comments = []
        for span in spans:
            start, end = span
            comment = text[start:end].strip()
            satd_comments.append(comment)

        # Count occurrences of each SATD comment
        comment_counts = Counter(satd_comments)

        # Get the top 10 most common SATD comments
        top_satd_comments = comment_counts.most_common(10)

        # Write summary entry
        summary_entry = {
            "file_name": filename,
            "satd_count": count,
            "top_satd_comments": [{"comment": comment, "count": count} for comment, count in top_satd_comments],
        }
        summary_file.write(json.dumps(summary_entry) + "\n")

        # Write full code to separate file
        # Replace unsafe characters from filename if needed
        # safe_filename = filename.replace("/", "__").replace("\\", "__")
        # with open(os.path.join(code_output_dir, f"{safe_filename}"), "w", encoding="utf-8") as code_file:
        #     code_file.write(text)

In [ ]:
sum_randoop = sum(top100.select(range(14))["satd_count"])
print(f"Total SATD count in top 100 files: {sum_randoop}")

In [4]:
# Get all entries with SATD count > 0
ds_with_satd = ds_all.filter(lambda x: x["satd_count"] > 0)
print(f"Total files with SATD: {len(ds_with_satd)}")

Total files with SATD: 431197


In [ ]:
# look for all files with a particular SATD comment
NON_SATD_COMMENTS = [
    "Regression assertion (captures the current behavior of the code)",
    "the probability of lineage i being in state j is p[i*nr_states +j]",
    "public static final String XXXbid",
]

# Get all entries with these comments
ds_non_satd = ds_all.filter(
    lambda x: any(comment in x["content"] for comment in NON_SATD_COMMENTS) if x["satd_count"] > 0 else False
)
print(f"Total files with non-SATD comments: {len(ds_non_satd)}")

Filter:   0%|          | 0/5168193 [00:00<?, ? examples/s]

Total files with non-SATD comments: 52


In [9]:
non_satd_file_ids = []
for entry in ds_non_satd:
    id = entry["id"]
    non_satd_file_ids.append(id)

In [ ]:
non_satd_output_dir = f"./investigation/non_satd_files"
os.makedirs(non_satd_output_dir, exist_ok=True)
for entry in ds_non_satd:
    filename = entry["file_name"]
    text = entry["content"]
    count = entry["satd_count"]
    spans = entry["satd_spans"]

    # Write full code to separate file
    # Replace unsafe characters from filename if needed
    safe_filename = filename.replace("/", "__").replace("\\", "__")
    with open(os.path.join(non_satd_output_dir, f"{safe_filename}"), "w", encoding="utf-8") as code_file:
        code_file.write(text)

In [ ]:
ds_with_satd_cleaned = ds_with_satd.filter(
    lambda x: not x["file_name"] in non_satd_file_ids
)

In [10]:
print(non_satd_file_ids)
# save the non_satd file names to json
with open("./investigation/non_satd_file_ids.json", "w") as f:
    json.dump(non_satd_file_ids, f)

# load the non_satd file names from json
with open("./investigation/non_satd_file_ids.json", "r") as f:
    non_satd_file_ids = json.load(f)

non_satd_file_ids

[1256864, 1256871, 1256880, 1982989, 1985996, 2186072, 2621387, 2621388, 2621389, 2621390, 2621391, 2621392, 2621393, 2621394, 2621395, 2621396, 2621398, 2621399, 2621400, 2621401, 2621402, 2621403, 2621404, 2621406, 2621407, 2621408, 2621410, 2621411, 2621412, 2621413, 2621415, 2621416, 2621417, 2621418, 2621419, 2621420, 2621421, 2621422, 2621423, 2877349, 2877627, 2877628, 2877629, 2877630, 2877631, 2877632, 2877633, 2877635, 2877955, 2878774, 2879761, 2879768]


[1256864,
 1256871,
 1256880,
 1982989,
 1985996,
 2186072,
 2621387,
 2621388,
 2621389,
 2621390,
 2621391,
 2621392,
 2621393,
 2621394,
 2621395,
 2621396,
 2621398,
 2621399,
 2621400,
 2621401,
 2621402,
 2621403,
 2621404,
 2621406,
 2621407,
 2621408,
 2621410,
 2621411,
 2621412,
 2621413,
 2621415,
 2621416,
 2621417,
 2621418,
 2621419,
 2621420,
 2621421,
 2621422,
 2621423,
 2877349,
 2877627,
 2877628,
 2877629,
 2877630,
 2877631,
 2877632,
 2877633,
 2877635,
 2877955,
 2878774,
 2879761,
 2879768]